# Train the bag damage model (Google Colab)
Teaches the model to find **tear, scratch, stain, stitching, hole** (+ bag).
1. **Runtime > Change runtime type > T4 GPU**
2. Run the cells top to bottom (roughly 30-90 minutes).
3. The last cell downloads `best.pt`. Put it in the project's `models/` folder.

Before you start, download your dataset zips (YOLOv8 format) to your computer. RUN_GUIDE.md, Part C has the links.

## 1. Upload the project zip (bag-tear-detector.zip)

In [ ]:
from google.colab import files
up = files.upload()            # choose bag-tear-detector.zip
import zipfile
zipfile.ZipFile(list(up)[0]).extractall('/content')
%cd /content/bag-tear-detector
!pip -q install ultralytics pyyaml

## 2. Upload ALL dataset zips (select several files at once)
Each zip is unpacked into its own folder under `data/raw/`. Roboflow zips (YOLOv8 format) already contain `data.yaml`.
The Mendeley bag zip may not: the cell below then creates a `classes.txt` for it (check the order against the dataset page).

In [ ]:
from google.colab import files
import zipfile, pathlib
up = files.upload()
for name in up:
    target = pathlib.Path('data/raw') / pathlib.Path(name).stem.replace(' ', '_')
    zipfile.ZipFile(name).extractall(target)
    has_names = any(target.rglob('data.yaml')) or any(target.rglob('classes.txt'))
    if not has_names:
        (target / 'classes.txt').write_text('damage\nbag\n')   # EDIT if the real class order differs
        print(name, '-> no data.yaml found, wrote classes.txt (damage, bag). CHECK THE ORDER.')
    else:
        print(name, '-> ok')

## 3. See which classes each dataset has and how they will be mapped
Lines ending `(ignored ...)` are dropped. If a class you want shows as ignored, add one line for it under `map:` in `configs/class_map.yaml`
(use the file browser on the left to edit it), then re-run this cell.

In [ ]:
!python training/prepare_dataset.py --src "data/raw/*" --list

## 4. Merge + clean
Renames classes, splits by original photo, and **warns about damage types with no or few boxes** (a type with no boxes can never be detected).

In [ ]:
!python training/prepare_dataset.py --src "data/raw/*" --out data/bag_tear

## 5. Train (YOLO11n, pretrained start)

In [ ]:
!python training/train.py --data data/bag_tear/data.yaml --epochs 80 --imgsz 640

## 6. Look at the results

In [ ]:
from IPython.display import Image, display
import os
for f in ('runs/bag_tear/results.png', 'runs/bag_tear/confusion_matrix_normalized.png', 'runs/bag_tear/val_batch0_pred.jpg'):
    if os.path.exists(f):
        display(Image(f))
    else:
        print('not found:', f)

## 7. Download the weights
Put the file in the project folder `models/best.pt`.

In [ ]:
from google.colab import files
files.download('models/best.pt')